In [7]:
import json
from pathlib import Path

import pandas as pd
from lakehouse import LakeConfig, connect, objects

lake = LakeConfig.from_env()
con = connect()                                  # DuckDB em memória, já autenticado no lake

EXPORTS = Path("/home/jovyan/exports")           # = ./data/exports no host
POWERBI = EXPORTS / "powerbi"

# Análise gold_export_powerbi

In [9]:
manifest = json.loads((POWERBI / "_manifest.json").read_text())
print("Exportado em (UTC):", manifest["exported_at_utc"])
pd.DataFrame([{"dataset": name, "rows": info["rows"]} for name, info in manifest["datasets"].items()])

Exportado em (UTC): 2026-11-20T22:36:43Z


,dataset,rows
0,gld_dim_campaign_running,9
1,gld_dim_customer,7888952
2,gld_dim_date,2558
3,gld_dim_event,34
4,gld_dim_holiday,3689
5,gld_dim_market,35
6,gld_dim_origin,7
7,gld_dim_product,246824
8,gld_dim_type,4
9,gld_fact_event_paid_media,23782


In [13]:
SOURCE = "host"   # "host" = ./data/exports/powerbi (o que o Power BI lê); "lake" = s3://gold (saída do dbt)

for file in sorted(POWERBI.glob("gld_*.parquet")):
    path = file.as_posix() if SOURCE == "host" else lake.uri("gold", file.name)
    con.execute(f"CREATE OR REPLACE VIEW {file.stem} AS SELECT * FROM read_parquet('{path}')")

df= con.sql("SELECT table_name FROM information_schema.tables WHERE table_type = 'VIEW' ORDER BY 1").df()

,table_name
0,gld_dim_campaign_running
1,gld_dim_customer
2,gld_dim_date
3,gld_dim_event
4,gld_dim_holiday
5,gld_dim_market
6,gld_dim_origin
7,gld_dim_product
8,gld_dim_type
9,gld_fact_event_paid_media


In [14]:
con.sql("""
SELECT d.year_month, m.mark_cod,
    sum(f.amount_fixed_rate_w_disc_wo_vat) FILTER (WHERE t.type = 'SALE')       AS gross_sales,
    sum(f.amount_fixed_rate_w_disc_wo_vat) FILTER (WHERE t.type = 'RETURN')     AS returns,
    sum(f.amount_fixed_rate_w_disc_wo_vat) FILTER (WHERE t.type = 'CANCELLED')  AS cancellations,
    sum(f.amount_fixed_rate_w_disc_wo_vat)                                      AS net_sales,
    sum(f.prod_qty) FILTER (WHERE t.type = 'SALE')                              AS units_sold,
    count(DISTINCT f.order_id) FILTER (WHERE t.type = 'SALE')                   AS orders,
    round(sum(f.amount_fixed_rate_w_disc_wo_vat)
          / count(DISTINCT f.order_id) FILTER (WHERE t.type = 'SALE'), 2)      AS avg_order_value,
    round(-sum(f.amount_fixed_rate_w_disc_wo_vat) FILTER (WHERE t.type = 'RETURN')
          / sum(f.amount_fixed_rate_w_disc_wo_vat) FILTER (WHERE t.type = 'SALE'), 4) AS return_rate
FROM gld_fact_sales AS f
JOIN gld_dim_date   AS d ON d.date_key = f.order_date_key
JOIN gld_dim_market AS m ON m.mark_sk  = f.mark_sk
JOIN gld_dim_type   AS t ON t.type_sk  = f.type_sk
WHERE m.mark_cod = 'Online ES' AND d.year_number = 2025
GROUP BY d.year_month, m.mark_cod
ORDER BY d.year_month
""").df()

,year_month,mark_cod,gross_sales,returns,cancellations,net_sales,units_sold,orders,avg_order_value,return_rate
0,2025-01,Online ES,2.563509e+06,-652668.48246,-19052.77272,1.891788e+06,335426.0,133692,14.15,0.2546
1,2025-02,Online ES,1.757761e+06,-469544.30862,-13527.91338,1.274688e+06,225585.0,94276,13.52,0.2671
2,2025-03,Online ES,1.388143e+06,-407846.89092,-9299.29548,9.709965e+05,113947.0,63245,15.35,0.2938
3,2025-04,Online ES,1.392700e+06,-400814.84436,-3406.63356,9.884783e+05,104949.0,62985,15.69,0.2878
4,2025-05,Online ES,1.674031e+06,-499516.02792,-5784.76530,1.168730e+06,130524.0,76731,15.23,0.2984
5,2025-06,Online ES,2.324852e+06,-717208.87836,-11973.60114,1.595669e+06,201401.0,100568,15.87,0.3085
6,2025-07,Online ES,2.464173e+06,-659916.04230,-22863.19296,1.781394e+06,272902.0,125504,14.19,0.2678
7,2025-08,Online ES,1.487384e+06,-373153.68930,-14968.63422,1.099262e+06,160074.0,80081,13.73,0.2509
8,2025-09,Online ES,1.621954e+06,-424161.17250,-5938.90194,1.191854e+06,112159.0,69922,17.05,0.2615
9,2025-10,Online ES,1.758855e+06,-493263.64434,-8811.37776,1.256780e+06,110618.0,70681,17.78,0.2804


In [15]:
con.sql("""
SELECT c.campaign_label,
       sum(f.amount_fixed_rate_w_disc_wo_vat)                                  AS net_sales,
       round(100 * sum(f.amount_fixed_rate_w_disc_wo_vat)
             / sum(sum(f.amount_fixed_rate_w_disc_wo_vat)) OVER (), 2)         AS pct_net_sales
FROM gld_fact_sales AS f
JOIN gld_dim_campaign_running AS c ON c.campaign_running_sk = f.campaign_running_sk
GROUP BY c.campaign_label
ORDER BY net_sales DESC
""").df()

,campaign_label,net_sales,pct_net_sales
0,Paid Media + Event,4.253122e+07,66.87
1,Paid Media,1.349672e+07,21.22
2,No,5.392225e+06,8.48
3,Paid Media + Event + Holiday,1.273312e+06,2.00
4,Paid Media + Holiday,5.961070e+05,0.94
5,Event,1.862834e+05,0.29
6,Holiday,1.189363e+05,0.19
7,Event + Holiday,1.092623e+04,0.02
8,Unknown,-3.317100e+03,-0.01


In [16]:
con.sql("""
SELECT d.year_month, m.mark_cod,
       sum(e.amount_paid_media)                                                   AS paid_media,
       count(DISTINCT e.date_key) FILTER (WHERE e.event_sk <> 0)                  AS days_with_event,
       count(DISTINCT e.date_key) FILTER (WHERE e.holiday_sk <> 0)                AS days_with_holiday,
       string_agg(DISTINCT ev.event, ', ') FILTER (WHERE e.event_sk <> 0)         AS events,
       string_agg(DISTINCT h.holiday_name, ', ') FILTER (WHERE e.holiday_sk <> 0) AS holidays
FROM gld_fact_event_paid_media AS e
JOIN gld_dim_date    AS d  ON d.date_key   = e.date_key
JOIN gld_dim_market  AS m  ON m.mark_sk    = e.mark_sk
JOIN gld_dim_event   AS ev ON ev.event_sk  = e.event_sk
JOIN gld_dim_holiday AS h  ON h.holiday_sk = e.holiday_sk
WHERE m.mark_cod = 'Online ES' AND d.year_month = '2025-11'
GROUP BY d.year_month, m.mark_cod
""").df()

,year_month,mark_cod,paid_media,days_with_event,days_with_holiday,events,holidays
0,2025-11,Online ES,414033.51,10,1,"stock_1, black_friday_get_your_basket_ready, b...",All Saints Day


# gold_scd2_simulation

In [17]:
for (state_file,) in con.execute(
        f"SELECT file FROM glob('{lake.uri('gold', '_simulation')}/*/_state.json') ORDER BY file").fetchall():
    state = objects.read_json(state_file)
    print(f"Cenário {state['scenario']} | atualizado {state['updated_at_utc']} | {len(state['history'])} mudança(s)")
    for change in state["history"]:
        key = change.get("prod_cod", change.get("customer_id"))
        print(f"   {change['at_utc']}  {change['dimension']}  {key}  {change['changes']}"
              f"  desde {change['effective_date']}  -> {change['fact_rows_rekeyed']} linhas da fato movidas")

Cenário demo | atualizado 2026-11-20T22:35:27Z | 1 mudança(s)
   2026-11-20T22:35:27Z  gld_dim_product  241859_BNL  {'fashiontype': 'Basic'}  desde 2025-11-15  -> 1561 linhas da fato movidas
Cenário exemplo_cliente | atualizado 2026-11-20T21:30:14Z | 1 mudança(s)
   2026-11-20T21:30:14Z  gld_dim_customer  1100724  {'subscription_date': '2025-06-01', 'unsubscription_date': '2025-03-01'}  desde None  -> 27 linhas da fato movidas
Cenário exemplo_produto | atualizado 2026-11-20T21:29:49Z | 2 mudança(s)
   2026-11-20T21:29:36Z  gld_dim_product  241859_BNL  {'fashiontype': 'Basic'}  desde 2025-11-15  -> 1561 linhas da fato movidas
   2026-11-20T21:29:49Z  gld_dim_product  241859_BNL  {'active': False}  desde 2025-12-10  -> 672 linhas da fato movidas


In [18]:
SCENARIO = "demo"

state = objects.read_json(lake.uri("gold", "_simulation", SCENARIO, "_state.json"))
last_change = state["history"][-1]
PROD_COD = last_change["prod_cod"]

con.execute(f"CREATE OR REPLACE VIEW sim_dim_product AS SELECT * FROM read_parquet('{state['latest']['gld_dim_product']}')")
con.execute(f"CREATE OR REPLACE VIEW sim_fact_sales  AS SELECT * FROM read_parquet('{state['latest']['gld_fact_sales']}')")
# Mesmos arquivos no host: EXPORTS / "scd2_simulation" / SCENARIO / "gld_dim_product.parquet" (e gld_fact_sales.parquet)

print("Produto:", PROD_COD)
print("Ações:", last_change["actions"])

Produto: 241859_BNL
Ações: ["type 2: closed prod_sk 202147 on 2025-11-14 and opened a new version of prod_spk 6137046 from 2025-11-15 with {'fashiontype': 'Basic'}"]


In [19]:
con.execute("""
SELECT 'gold (dbt)' AS origem, prod_sk, prod_spk, fashiontype, active, valid_from, valid_to, is_current
FROM gld_dim_product WHERE prod_cod = $prod_cod
UNION ALL
SELECT 'simulação', prod_sk, prod_spk, fashiontype, active, valid_from, valid_to, is_current
FROM sim_dim_product WHERE prod_cod = $prod_cod
ORDER BY origem, prod_spk, valid_from
""", {"prod_cod": PROD_COD}).df()

,origem,prod_sk,prod_spk,fashiontype,active,valid_from,valid_to,is_current
0,gold (dbt),151099,5974102,Fashion Basic,False,1900-01-01,9999-12-31,False
1,gold (dbt),202147,6137046,Fashion Basic,True,1900-01-01,9999-12-31,True
2,simulação,151099,5974102,Fashion Basic,False,1900-01-01,9999-12-31,False
3,simulação,202147,6137046,Fashion Basic,True,1900-01-01,2025-11-14,False
4,simulação,246824,6137046,Basic,True,2025-11-15,9999-12-31,True


In [20]:
con.execute("""
SELECT p.prod_sk, p.prod_spk, p.fashiontype, p.valid_from, p.valid_to,
       count(*)                                AS fact_rows,
       count(DISTINCT f.order_id)              AS orders,
       sum(f.prod_qty)                         AS units,
       sum(f.amount_fixed_rate_w_disc_wo_vat)  AS net_sales,
       min(d.calendar_date)                    AS first_order,
       max(d.calendar_date)                    AS last_order
FROM sim_fact_sales  AS f
JOIN sim_dim_product AS p ON p.prod_sk  = f.prod_sk
JOIN gld_dim_date    AS d ON d.date_key = f.order_date_key
WHERE p.prod_cod = $prod_cod
GROUP BY ALL
ORDER BY p.prod_spk, p.valid_from
""", {"prod_cod": PROD_COD}).df()

,prod_sk,prod_spk,fashiontype,valid_from,valid_to,fact_rows,orders,units,net_sales,first_order,last_order
0,202147,6137046,Fashion Basic,1900-01-01,2025-11-14,1264,1006,748.0,38321.41874,2025-09-20,2025-11-14
1,246824,6137046,Basic,2025-11-15,9999-12-31,1561,1384,1208.0,62826.05528,2025-11-15,2025-12-31


In [21]:
con.execute("""
SELECT 'gold (dbt)' AS origem, p.fashiontype, count(DISTINCT f.order_id) AS orders,
       sum(f.amount_fixed_rate_w_disc_wo_vat) AS net_sales
FROM gld_fact_sales AS f JOIN gld_dim_product AS p ON p.prod_sk = f.prod_sk
WHERE p.prod_cod = $prod_cod
GROUP BY ALL
UNION ALL
SELECT 'simulação', p.fashiontype, count(DISTINCT f.order_id),
       sum(f.amount_fixed_rate_w_disc_wo_vat)
FROM sim_fact_sales AS f JOIN sim_dim_product AS p ON p.prod_sk = f.prod_sk
WHERE p.prod_cod = $prod_cod
GROUP BY ALL
ORDER BY origem, fashiontype
""", {"prod_cod": PROD_COD}).df()

,origem,fashiontype,orders,net_sales
0,gold (dbt),Fashion Basic,2390,101147.47402
1,simulação,Basic,1384,62826.05528
2,simulação,Fashion Basic,1006,38321.41874


In [ ]:
con.sql("""
SELECT
    (SELECT count(*) FROM gld_fact_sales) = (SELECT count(*) FROM sim_fact_sales)                       AS same_rows,
    (SELECT sum(amount_fixed_rate_w_disc_wo_vat) FROM gld_fact_sales)
        = (SELECT sum(amount_fixed_rate_w_disc_wo_vat) FROM sim_fact_sales)                              AS same_net_sales,
    (SELECT count(*) FROM sim_fact_sales AS f ANTI JOIN sim_dim_product AS p ON p.prod_sk = f.prod_sk)  AS orphan_rows,
    (SELECT count(*) FROM (SELECT prod_cod FROM sim_dim_product
                           WHERE is_current AND NOT is_inferred AND prod_sk <> 0
                           GROUP BY prod_cod HAVING count(*) > 1))                                      AS skus_with_2_current
""").df()